## Imports

In [56]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error


## Load Dataset

In [57]:
df = pd.read_csv("/Users/ks/PycharmProjects/INF_172_Financial_Health_Planning/Backend/data/NFCS_2024_State_Data_250623.csv")

df.head()


,NFCSID,STATEQ,CENSUSDIV,CENSUSREG,A50A,A3Ar_w,A50B,A4A_new_w,A5_2015,A6,...,M6,M7,M8,M31,M50,M9,M10,wgt_n2,wgt_d2,wgt_s3
0,2024010001,36,3,2,2,3,9,1,6,1,...,1,3,98,98,98,1,2,1.153548,1.123949,0.859644
1,2024010002,48,9,4,1,6,6,1,5,5,...,1,3,98,3,3,1,2,1.398688,0.863440,0.975078
2,2024010003,38,9,4,1,6,6,1,4,1,...,1,3,1,98,98,2,2,1.398688,0.472645,0.893974
3,2024010004,48,9,4,2,6,12,1,7,1,...,1,3,1,4,98,1,2,1.250293,0.614156,0.778748
4,2024010005,44,7,3,2,6,12,2,7,4,...,1,3,2,3,2,1,1,1.250076,2.228957,0.783507


## Create Target Variable (Recommended Contribution)

In [58]:
def estimate_contribution(row):
    income = row["A8_2021"]
    risk = row["H30_3"] if "H30_3" in row else 0
    chronic = row.get("H1", 0)
    hospitalizations = row.get("H30_3", 0)

    base = 20
    base += (income * 5)
    base += (risk * 15)
    base += (chronic * 20)
    base += (hospitalizations * 25)

    return max(base, 10)

df["recommended_contribution"] = df.apply(estimate_contribution, axis=1)

print(df[["recommended_contribution"]].describe())


       recommended_contribution
count              25539.000000
mean                 248.746035
std                  663.666712
min                   85.000000
25%                  125.000000
50%                  145.000000
75%                  155.000000
max                 6000.000000


## Feature Selection

In [60]:
feature_cols = [
    "A5_2015",      # education level
    "A9",           # employment status
    "J1",           # financial satisfaction
    "J2",           # risk tolerance
    "J4",           # difficulty paying bills
    "J5",           # emergency savings
    "H1",           # health insurance
    "F1",           # credit cards
    "G23",          # debt perception
    "wgt_n2"        # sample weight
]

# Keep only columns that exist in dataset
feature_cols = [col for col in feature_cols if col in df.columns]

# df = df.convert_dtypes()

X = df[feature_cols]
y = df["recommended_contribution"]

## Preprocessing Pipeline

In [61]:
categorical = [
    "A5_2015",
    "A9",
    "J4",
    "J5",
    "H1"]

numeric = [
    "J1",
    "J2",
    "F1",
    "G23",
    "wgt_n2"]
# categorical = X.select_dtypes(include=["object"]).columns.tolist()
# numeric = [col for col in X.columns if col not in categorical]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
        ("num", "passthrough", numeric)
    ],
    remainder="drop"
)

## Model Pipeline
- Random Forest

In [62]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=200,
        random_state=42
    ))
])


### Train/Test Split

In [63]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)


### Train Model

In [64]:
model.fit(X_train, y_train)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['A5_2015', 'A9', 'J4', 'J5',
                                                   'H1']),
                                                 ('num', 'passthrough',
                                                  ['J1', 'J2', 'F1', 'G23',
                                                   'wgt_n2'])])),
                ('regressor',
                 RandomForestRegressor(n_estimators=200, random_state=42))])

## Evaluate Performance

In [65]:
val_pred = model.predict(X_val)
mae = mean_absolute_error(y_val, val_pred)

print(f"Validation MAE: {mae:.2f}")


Validation MAE: 163.47


In [66]:
# Get fitted pipeline pieces
preprocessor = model.named_steps["preprocessor"]
rf = model.named_steps["regressor"]

# OneHotEncoder (fitted)
ohe = preprocessor.named_transformers_["cat"]

# Feature names after one-hot encoding
cat_features = ohe.get_feature_names_out()

# Numeric feature names
num_features = preprocessor.transformers_[1][2]

# Combine
feature_names = list(cat_features) + list(num_features)

# Importance scores
importances = rf.feature_importances_

fi = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

print(fi.head(20))

      feature  importance
26      H1_98    0.243934
32     wgt_n2    0.151193
27      H1_99    0.126502
28         J1    0.067185
29         J2    0.065388
30         F1    0.062858
31        G23    0.062012
1   A5_2015_2    0.016505
16       J4_2    0.013642
3   A5_2015_4    0.013632
8        A9_2    0.011296
17       J4_3    0.011065
21       J5_2    0.010054
20       J5_1    0.010018
4   A5_2015_5    0.010014
22      J5_98    0.009771
5   A5_2015_6    0.009685
10       A9_4    0.008964
9        A9_3    0.008704
15       J4_1    0.008550


In [34]:
joblib.dump(model, "../models/hsa_recommendation_model.pkl")

print("Model saved as hsa_recommendation_model.pkl")

Model saved as hsa_recommendation_model.pkl
